In [ ]:
import pandas as pd
import numpy as np

# 1. Carrega o arquivo de vendas original
df = pd.read_csv("sales_messy_synthetic_updated.csv")

# 2. Remove espaços extras do início, fim e duplicados de todas as colunas de texto
df = df.map(lambda x: " ".join(x.split()) if isinstance(x, str) else x)

# 3. Remove os 35 IDs de pedidos duplicados (order_id), mantendo a primeira ocorrência
df.drop_duplicates(subset=['order_id'], keep='first', inplace=True)

# 4. Remove colunas que estejam 100% completamente vazias
df.dropna(axis=1, how='all', inplace=True)

# 5. Remove linhas que estejam 100% completamente vazias
df.dropna(axis=0, how='all', inplace=True)

# 6. Padroniza todas as colunas de texto para terem a primeira letra maiúscula (corrige "east" para "East")
colunas_texto = df.select_dtypes(include=['object', 'string']).columns
df[colunas_texto] = df[colunas_texto].apply(lambda x: x.str.title())

# 7. Corrige padrões de Sim/Não na coluna 'is_priority'
df['is_priority'] = df['is_priority'].replace({'Y': 'Yes', 'N': 'No', 'True': 'Yes', 'False': 'No', True: 'Yes', False: 'No'})

# 8. Transforma qualquer texto vazio ("  ") de todas as colunas em NaN (nulo real)
df = df.replace(r'^\s*$', np.nan, regex=True)

# 9. Limpa o símbolo de '%' da coluna de descontos e corrige a escala dividindo por 100
tem_porcentagem = df['discount'].astype(str).str.contains('%', na=False)
df['discount'] = df['discount'].astype(str).str.replace('%', '', regex=False)
df['discount'] = pd.to_numeric(df['discount'], errors='coerce')
df.loc[tem_porcentagem, 'discount'] = df.loc[tem_porcentagem, 'discount'] / 100

# 10. Limpa a coluna 'unit_price': remove o texto 'Confidential' e elimina preços negativos (< 0)
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df.loc[df['unit_price'] < 0, 'unit_price'] = np.nan

# 11. Corrige as datas da coluna 'order_date' (converte texto inválido e datas impossíveis como 31/02 para NaN)
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce')

# 12. Limpa e força a coluna 'customer_age' para o tipo numérico (tirando a palavra texto 'nan')
df['customer_age'] = pd.to_numeric(df['customer_age'].astype(str).str.strip(), errors='coerce')



In [ ]:
# 1. importing the file, creating a copy for safety and reading the file.

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("sales_messy_synthetic_updated.csv")
df_original = df.copy
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1258 entries, 0 to 1257
Data columns (total 36 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   order_id                   1228 non-null   str    
 1   customer_id                1228 non-null   str    
 2   customer_name              1243 non-null   str    
 3   sales_rep                  1194 non-null   str    
 4   region                     1221 non-null   str    
 5   country                    1204 non-null   str    
 6   sales_channel              1228 non-null   str    
 7   product_id                 1228 non-null   str    
 8   product_name               1228 non-null   str    
 9   category                   1216 non-null   str    
 10  order_date                 1216 non-null   str    
 11  quantity                   1220 non-null   str    
 12  unit_price                 1195 non-null   str    
 13  discount                   1180 non-null   str    
 14  shi

In [ ]:
# 2. Remove the extra spaces at the start and in the end and gets rid of the duplicated rows of all string columns.

In [5]:
df = df.map(lambda x: " ".join(x.split()) if isinstance(x, str) else x)
df

,order_id,customer_id,customer_name,sales_rep,region,country,sales_channel,product_id,product_name,category,...,discount_amount,net_sales,profit,completely_empty_column_1,completely_empty_column_2,mostly_empty_notes,blank_column_1,Unnamed: blank_column,system_audit_dummy,internal_notes
0,ORD-10450,CUST-1100,Isabela Nogueira,Emma Wilson,North,France,ONLINE,PROD-124,Desk,Electronics,...,2492.74688,9265.49312,9221.86312,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ORD-10755,CUST-1011,Rafael Teixeira,Carla Davis,North,France,Retail,PROD-120,Tablet,Food,...,220.13210,5572.81790,5569.99790,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ORD-10531,CUST-1009,Rafael Costa,Daniel Brown,East,Germany,Wholesale,PROD-119,Chair,Office Supplies,...,5570.18202,19520.72798,19416.17798,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ORD-10377,CUST-1094,Camila Oliveira,Emma Wilson,South,UK,Online,PROD-134,Headphones,Office Supplies,...,1887.38732,5155.10268,5137.79268,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ORD-10998,CUST-1191,Larissa Monteiro,Grace Taylor,South,UK,Online,PROD-140,Tablet,Food,...,231.17430,6373.80570,6310.48570,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253,ORD-10475,CUST-1318,Luana Silva,Grace Taylor,East,USA,Direct,PROD-144,Tablet,Office Supplies,...,1595.50930,5723.34070,5628.38070,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1254,ORD-10225,CUST-1112,Diego Carvalho,Henry Moore,East,Brazil,Direct,PROD-104,Tablet,Furniture,...,4134.68832,11066.37168,10972.99168,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1255,ORD-10481,CUST-1116,Bianca Teixeira,Bob Smith,East,USA,Wholesale,PROD-133,Laptop,ELECTRONICS,...,2588.30532,6688.77468,6564.82468,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1256,ORD-10854,CUST-1305,Rodrigo Nogueira,Emma Wilson,Central,Brazil,Online,PROD-113,Desk,Office Supplies,...,1618.93280,7793.46720,7657.03720,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 3. Remove the 35 IDs of duplicated orders (order_id), keeping the first occurrence

In [6]:
df.drop_duplicates(subset=['order_id'], keep='first', inplace=True)
df

,order_id,customer_id,customer_name,sales_rep,region,country,sales_channel,product_id,product_name,category,...,discount_amount,net_sales,profit,completely_empty_column_1,completely_empty_column_2,mostly_empty_notes,blank_column_1,Unnamed: blank_column,system_audit_dummy,internal_notes
0,ORD-10450,CUST-1100,Isabela Nogueira,Emma Wilson,North,France,ONLINE,PROD-124,Desk,Electronics,...,2492.74688,9265.49312,9221.86312,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ORD-10755,CUST-1011,Rafael Teixeira,Carla Davis,North,France,Retail,PROD-120,Tablet,Food,...,220.13210,5572.81790,5569.99790,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ORD-10531,CUST-1009,Rafael Costa,Daniel Brown,East,Germany,Wholesale,PROD-119,Chair,Office Supplies,...,5570.18202,19520.72798,19416.17798,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ORD-10377,CUST-1094,Camila Oliveira,Emma Wilson,South,UK,Online,PROD-134,Headphones,Office Supplies,...,1887.38732,5155.10268,5137.79268,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ORD-10998,CUST-1191,Larissa Monteiro,Grace Taylor,South,UK,Online,PROD-140,Tablet,Food,...,231.17430,6373.80570,6310.48570,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253,ORD-10475,CUST-1318,Luana Silva,Grace Taylor,East,USA,Direct,PROD-144,Tablet,Office Supplies,...,1595.50930,5723.34070,5628.38070,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1254,ORD-10225,CUST-1112,Diego Carvalho,Henry Moore,East,Brazil,Direct,PROD-104,Tablet,Furniture,...,4134.68832,11066.37168,10972.99168,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1255,ORD-10481,CUST-1116,Bianca Teixeira,Bob Smith,East,USA,Wholesale,PROD-133,Laptop,ELECTRONICS,...,2588.30532,6688.77468,6564.82468,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1256,ORD-10854,CUST-1305,Rodrigo Nogueira,Emma Wilson,Central,Brazil,Online,PROD-113,Desk,Office Supplies,...,1618.93280,7793.46720,7657.03720,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 4. Remove columns and rows that are 100% empty.

In [7]:
df.dropna(axis=1, how='all', inplace=True)
df.dropna(axis=0, how='all', inplace=True)
df

,order_id,customer_id,customer_name,sales_rep,region,country,sales_channel,product_id,product_name,category,...,warehouse,is_priority,delivery_days,return_reason,gross_sales,discount_amount,net_sales,profit,mostly_empty_notes,internal_notes
0,ORD-10450,CUST-1100,Isabela Nogueira,Emma Wilson,North,France,ONLINE,PROD-124,Desk,Electronics,...,WH-01,Yes,8.0,NaN,11758.24,2492.74688,9265.49312,9221.86312,NaN,NaN
1,ORD-10755,CUST-1011,Rafael Teixeira,Carla Davis,North,France,Retail,PROD-120,Tablet,Food,...,WH-04,No,3.0,Wrong Item,5792.95,220.13210,5572.81790,5569.99790,NaN,NaN
2,ORD-10531,CUST-1009,Rafael Costa,Daniel Brown,East,Germany,Wholesale,PROD-119,Chair,Office Supplies,...,WH-04,Yes,13.0,Wrong Item,25090.91,5570.18202,19520.72798,19416.17798,NaN,NaN
3,ORD-10377,CUST-1094,Camila Oliveira,Emma Wilson,South,UK,Online,PROD-134,Headphones,Office Supplies,...,WH-03,No,6.0,Late Delivery,7042.49,1887.38732,5155.10268,5137.79268,NaN,NaN
4,ORD-10998,CUST-1191,Larissa Monteiro,Grace Taylor,South,UK,Online,PROD-140,Tablet,Food,...,WH-03,No,10.0,Customer Changed Mind,6604.98,231.17430,6373.80570,6310.48570,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253,ORD-10475,CUST-1318,Luana Silva,Grace Taylor,East,USA,Direct,PROD-144,Tablet,Office Supplies,...,WH-01,No,13.0,Late Delivery,7318.85,1595.50930,5723.34070,5628.38070,NaN,NaN
1254,ORD-10225,CUST-1112,Diego Carvalho,Henry Moore,East,Brazil,Direct,PROD-104,Tablet,Furniture,...,WH-03,Yes,1.0,NaN,15201.06,4134.68832,11066.37168,10972.99168,NaN,NaN
1255,ORD-10481,CUST-1116,Bianca Teixeira,Bob Smith,East,USA,Wholesale,PROD-133,Laptop,ELECTRONICS,...,WH-02,No,2.0,Customer Changed Mind,9277.08,2588.30532,6688.77468,6564.82468,NaN,NaN
1256,ORD-10854,CUST-1305,Rodrigo Nogueira,Emma Wilson,Central,Brazil,Online,PROD-113,Desk,Office Supplies,...,WH-02,Yes,1.0,Late Delivery,9412.40,1618.93280,7793.46720,7657.03720,NaN,NaN


In [ ]:
# 5. Keep columns only if they have at least 70% real data (drops if >30% is missing) and Keep rows only if they have at least 80% real data (drops if >20% is missing)


In [8]:
col_limit = int(len(df) * 0.70)
df.dropna(axis=1, thresh=col_limit, inplace=True)

row_limit = int(len(df.columns) * 0.80)
df.dropna(axis=0, thresh=row_limit, inplace=True)
df

,order_id,customer_id,customer_name,sales_rep,region,country,sales_channel,product_id,product_name,category,...,customer_age,customer_segment,warehouse,is_priority,delivery_days,return_reason,gross_sales,discount_amount,net_sales,profit
0,ORD-10450,CUST-1100,Isabela Nogueira,Emma Wilson,North,France,ONLINE,PROD-124,Desk,Electronics,...,27.0,Enterprise,WH-01,Yes,8.0,NaN,11758.24,2492.74688,9265.49312,9221.86312
1,ORD-10755,CUST-1011,Rafael Teixeira,Carla Davis,North,France,Retail,PROD-120,Tablet,Food,...,68.0,Consumer,WH-04,No,3.0,Wrong Item,5792.95,220.13210,5572.81790,5569.99790
2,ORD-10531,CUST-1009,Rafael Costa,Daniel Brown,East,Germany,Wholesale,PROD-119,Chair,Office Supplies,...,58.0,Corporate,WH-04,Yes,13.0,Wrong Item,25090.91,5570.18202,19520.72798,19416.17798
3,ORD-10377,CUST-1094,Camila Oliveira,Emma Wilson,South,UK,Online,PROD-134,Headphones,Office Supplies,...,61.0,Consumer,WH-03,No,6.0,Late Delivery,7042.49,1887.38732,5155.10268,5137.79268
4,ORD-10998,CUST-1191,Larissa Monteiro,Grace Taylor,South,UK,Online,PROD-140,Tablet,Food,...,24.0,Small Business,WH-03,No,10.0,Customer Changed Mind,6604.98,231.17430,6373.80570,6310.48570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253,ORD-10475,CUST-1318,Luana Silva,Grace Taylor,East,USA,Direct,PROD-144,Tablet,Office Supplies,...,44.0,Consumer,WH-01,No,13.0,Late Delivery,7318.85,1595.50930,5723.34070,5628.38070
1254,ORD-10225,CUST-1112,Diego Carvalho,Henry Moore,East,Brazil,Direct,PROD-104,Tablet,Furniture,...,56.0,Consumer,WH-03,Yes,1.0,NaN,15201.06,4134.68832,11066.37168,10972.99168
1255,ORD-10481,CUST-1116,Bianca Teixeira,Bob Smith,East,USA,Wholesale,PROD-133,Laptop,ELECTRONICS,...,49.0,Small Business,WH-02,No,2.0,Customer Changed Mind,9277.08,2588.30532,6688.77468,6564.82468
1256,ORD-10854,CUST-1305,Rodrigo Nogueira,Emma Wilson,Central,Brazil,Online,PROD-113,Desk,Office Supplies,...,68.0,Enterprise,WH-02,Yes,1.0,Late Delivery,9412.40,1618.93280,7793.46720,7657.03720


In [ ]:
# 6. Standardize all text columns to have the first letter capitalized (corrects "east" to "East")

In [9]:
colunas_texto = df.select_dtypes(include=['object', 'string']).columns
df[colunas_texto] = df[colunas_texto].apply(lambda x: x.str.title())
df

,order_id,customer_id,customer_name,sales_rep,region,country,sales_channel,product_id,product_name,category,...,customer_age,customer_segment,warehouse,is_priority,delivery_days,return_reason,gross_sales,discount_amount,net_sales,profit
0,Ord-10450,Cust-1100,Isabela Nogueira,Emma Wilson,North,France,Online,Prod-124,Desk,Electronics,...,27.0,Enterprise,Wh-01,Yes,8.0,NaN,11758.24,2492.74688,9265.49312,9221.86312
1,Ord-10755,Cust-1011,Rafael Teixeira,Carla Davis,North,France,Retail,Prod-120,Tablet,Food,...,68.0,Consumer,Wh-04,No,3.0,Wrong Item,5792.95,220.13210,5572.81790,5569.99790
2,Ord-10531,Cust-1009,Rafael Costa,Daniel Brown,East,Germany,Wholesale,Prod-119,Chair,Office Supplies,...,58.0,Corporate,Wh-04,Yes,13.0,Wrong Item,25090.91,5570.18202,19520.72798,19416.17798
3,Ord-10377,Cust-1094,Camila Oliveira,Emma Wilson,South,Uk,Online,Prod-134,Headphones,Office Supplies,...,61.0,Consumer,Wh-03,No,6.0,Late Delivery,7042.49,1887.38732,5155.10268,5137.79268
4,Ord-10998,Cust-1191,Larissa Monteiro,Grace Taylor,South,Uk,Online,Prod-140,Tablet,Food,...,24.0,Small Business,Wh-03,No,10.0,Customer Changed Mind,6604.98,231.17430,6373.80570,6310.48570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253,Ord-10475,Cust-1318,Luana Silva,Grace Taylor,East,Usa,Direct,Prod-144,Tablet,Office Supplies,...,44.0,Consumer,Wh-01,No,13.0,Late Delivery,7318.85,1595.50930,5723.34070,5628.38070
1254,Ord-10225,Cust-1112,Diego Carvalho,Henry Moore,East,Brazil,Direct,Prod-104,Tablet,Furniture,...,56.0,Consumer,Wh-03,Yes,1.0,NaN,15201.06,4134.68832,11066.37168,10972.99168
1255,Ord-10481,Cust-1116,Bianca Teixeira,Bob Smith,East,Usa,Wholesale,Prod-133,Laptop,Electronics,...,49.0,Small Business,Wh-02,No,2.0,Customer Changed Mind,9277.08,2588.30532,6688.77468,6564.82468
1256,Ord-10854,Cust-1305,Rodrigo Nogueira,Emma Wilson,Central,Brazil,Online,Prod-113,Desk,Office Supplies,...,68.0,Enterprise,Wh-02,Yes,1.0,Late Delivery,9412.40,1618.93280,7793.46720,7657.03720


In [ ]:
# 7. Fix Yes/No patterns in the 'is_priority' column

In [10]:
df["is_priority"].unique()

<StringArray>
['Yes', 'No', 'Y', 'N', 'False', 'True']
Length: 6, dtype: str

In [11]:
df['is_priority'] = df['is_priority'].replace({'Y': 'Yes', 'N': 'No', 'True': 'Yes', 'False': 'No', True: 'Yes', False: 'No'})
df["is_priority"].unique()

<StringArray>
['Yes', 'No']
Length: 2, dtype: str

In [ ]:
# 8. Transform any empty or whitespace-only cells in all columns to NaN (null)

In [12]:
df = df.replace(r'^\s*$', np.nan, regex=True)

In [ ]:
# 9. Clean the '%' symbol from the discount column and correct the scale by dividing by 100

In [13]:
tem_porcentagem = df['discount'].astype(str).str.contains('%', na=False)
df['discount'] = df['discount'].astype(str).str.replace('%', '', regex=False)
df['discount'] = pd.to_numeric(df['discount'], errors='coerce')
df.loc[tem_porcentagem, 'discount'] = df.loc[tem_porcentagem, 'discount'] / 100
df["discount"].unique()
df

,order_id,customer_id,customer_name,sales_rep,region,country,sales_channel,product_id,product_name,category,...,customer_age,customer_segment,warehouse,is_priority,delivery_days,return_reason,gross_sales,discount_amount,net_sales,profit
0,Ord-10450,Cust-1100,Isabela Nogueira,Emma Wilson,North,France,Online,Prod-124,Desk,Electronics,...,27.0,Enterprise,Wh-01,Yes,8.0,NaN,11758.24,2492.74688,9265.49312,9221.86312
1,Ord-10755,Cust-1011,Rafael Teixeira,Carla Davis,North,France,Retail,Prod-120,Tablet,Food,...,68.0,Consumer,Wh-04,No,3.0,Wrong Item,5792.95,220.13210,5572.81790,5569.99790
2,Ord-10531,Cust-1009,Rafael Costa,Daniel Brown,East,Germany,Wholesale,Prod-119,Chair,Office Supplies,...,58.0,Corporate,Wh-04,Yes,13.0,Wrong Item,25090.91,5570.18202,19520.72798,19416.17798
3,Ord-10377,Cust-1094,Camila Oliveira,Emma Wilson,South,Uk,Online,Prod-134,Headphones,Office Supplies,...,61.0,Consumer,Wh-03,No,6.0,Late Delivery,7042.49,1887.38732,5155.10268,5137.79268
4,Ord-10998,Cust-1191,Larissa Monteiro,Grace Taylor,South,Uk,Online,Prod-140,Tablet,Food,...,24.0,Small Business,Wh-03,No,10.0,Customer Changed Mind,6604.98,231.17430,6373.80570,6310.48570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253,Ord-10475,Cust-1318,Luana Silva,Grace Taylor,East,Usa,Direct,Prod-144,Tablet,Office Supplies,...,44.0,Consumer,Wh-01,No,13.0,Late Delivery,7318.85,1595.50930,5723.34070,5628.38070
1254,Ord-10225,Cust-1112,Diego Carvalho,Henry Moore,East,Brazil,Direct,Prod-104,Tablet,Furniture,...,56.0,Consumer,Wh-03,Yes,1.0,NaN,15201.06,4134.68832,11066.37168,10972.99168
1255,Ord-10481,Cust-1116,Bianca Teixeira,Bob Smith,East,Usa,Wholesale,Prod-133,Laptop,Electronics,...,49.0,Small Business,Wh-02,No,2.0,Customer Changed Mind,9277.08,2588.30532,6688.77468,6564.82468
1256,Ord-10854,Cust-1305,Rodrigo Nogueira,Emma Wilson,Central,Brazil,Online,Prod-113,Desk,Office Supplies,...,68.0,Enterprise,Wh-02,Yes,1.0,Late Delivery,9412.40,1618.93280,7793.46720,7657.03720


In [ ]:
# 10. Clean the 'unit_price' column: remove 'Confidential' text and eliminate negative prices (< 0)

In [14]:
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df.loc[df['unit_price'] < 0, 'unit_price'] = np.nan

In [ ]:
# 11. Fix dates in the 'order_date' column (convert invalid text and impossible dates like 31/02 to NaN)

In [15]:
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce')

In [ ]:
# 12. Clean and force the 'customer_age' column to numeric type (removing the text word 'nan')

In [17]:
df['customer_age'] = pd.to_numeric(df['customer_age'].astype(str).str.strip(), errors='coerce')
df

,order_id,customer_id,customer_name,sales_rep,region,country,sales_channel,product_id,product_name,category,...,customer_age,customer_segment,warehouse,is_priority,delivery_days,return_reason,gross_sales,discount_amount,net_sales,profit
0,Ord-10450,Cust-1100,Isabela Nogueira,Emma Wilson,North,France,Online,Prod-124,Desk,Electronics,...,27.0,Enterprise,Wh-01,Yes,8.0,NaN,11758.24,2492.74688,9265.49312,9221.86312
1,Ord-10755,Cust-1011,Rafael Teixeira,Carla Davis,North,France,Retail,Prod-120,Tablet,Food,...,68.0,Consumer,Wh-04,No,3.0,Wrong Item,5792.95,220.13210,5572.81790,5569.99790
2,Ord-10531,Cust-1009,Rafael Costa,Daniel Brown,East,Germany,Wholesale,Prod-119,Chair,Office Supplies,...,58.0,Corporate,Wh-04,Yes,13.0,Wrong Item,25090.91,5570.18202,19520.72798,19416.17798
3,Ord-10377,Cust-1094,Camila Oliveira,Emma Wilson,South,Uk,Online,Prod-134,Headphones,Office Supplies,...,61.0,Consumer,Wh-03,No,6.0,Late Delivery,7042.49,1887.38732,5155.10268,5137.79268
4,Ord-10998,Cust-1191,Larissa Monteiro,Grace Taylor,South,Uk,Online,Prod-140,Tablet,Food,...,24.0,Small Business,Wh-03,No,10.0,Customer Changed Mind,6604.98,231.17430,6373.80570,6310.48570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253,Ord-10475,Cust-1318,Luana Silva,Grace Taylor,East,Usa,Direct,Prod-144,Tablet,Office Supplies,...,44.0,Consumer,Wh-01,No,13.0,Late Delivery,7318.85,1595.50930,5723.34070,5628.38070
1254,Ord-10225,Cust-1112,Diego Carvalho,Henry Moore,East,Brazil,Direct,Prod-104,Tablet,Furniture,...,56.0,Consumer,Wh-03,Yes,1.0,NaN,15201.06,4134.68832,11066.37168,10972.99168
1255,Ord-10481,Cust-1116,Bianca Teixeira,Bob Smith,East,Usa,Wholesale,Prod-133,Laptop,Electronics,...,49.0,Small Business,Wh-02,No,2.0,Customer Changed Mind,9277.08,2588.30532,6688.77468,6564.82468
1256,Ord-10854,Cust-1305,Rodrigo Nogueira,Emma Wilson,Central,Brazil,Online,Prod-113,Desk,Office Supplies,...,68.0,Enterprise,Wh-02,Yes,1.0,Late Delivery,9412.40,1618.93280,7793.46720,7657.03720


In [ ]:
import pandas as pd
import numpy as np

# 1. Carrega o arquivo de vendas original
df = pd.read_csv("sales_messy_synthetic_updated.csv")

# 2. Remove espaços extras do início, fim e duplicados de todas as colunas de texto
df = df.map(lambda x: " ".join(x.split()) if isinstance(x, str) else x)

# 3. Remove os 35 IDs de pedidos duplicados (order_id), mantendo a primeira ocorrência
df.drop_duplicates(subset=['order_id'], keep='first', inplace=True)

# 4. Remove colunas que estejam 100% completamente vazias
df.dropna(axis=1, how='all', inplace=True)

# 5. Remove linhas que estejam 100% completamente vazias
df.dropna(axis=0, how='all', inplace=True)

# 6. Padroniza todas as colunas de texto para terem a primeira letra maiúscula (corrige "east" para "East")
colunas_texto = df.select_dtypes(include=['object', 'string']).columns
df[colunas_texto] = df[colunas_texto].apply(lambda x: x.str.title())

# 7. Corrige padrões de Sim/Não na coluna 'is_priority'
df['is_priority'] = df['is_priority'].replace({'Y': 'Yes', 'N': 'No', 'True': 'Yes', 'False': 'No', True: 'Yes', False: 'No'})

# 8. Transforma qualquer texto vazio ("  ") de todas as colunas em NaN (nulo real)
df = df.replace(r'^\s*$', np.nan, regex=True)

# 9. Limpa o símbolo de '%' da coluna de descontos e corrige a escala dividindo por 100
tem_porcentagem = df['discount'].astype(str).str.contains('%', na=False)
df['discount'] = df['discount'].astype(str).str.replace('%', '', regex=False)
df['discount'] = pd.to_numeric(df['discount'], errors='coerce')
df.loc[tem_porcentagem, 'discount'] = df.loc[tem_porcentagem, 'discount'] / 100

# 10. Limpa a coluna 'unit_price': remove o texto 'Confidential' e elimina preços negativos (< 0)
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df.loc[df['unit_price'] < 0, 'unit_price'] = np.nan

# 11. Corrige as datas da coluna 'order_date' (converte texto inválido e datas impossíveis como 31/02 para NaN)
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce')

# 12. Limpa e força a coluna 'customer_age' para o tipo numérico (tirando a palavra texto 'nan')
df['customer_age'] = pd.to_numeric(df['customer_age'].astype(str).str.strip(), errors='coerce')

